# Grid-Forming Converter Frequency Response — and a bridge to PyTorch

*A guided tour of the `PowerDynamicSimulator` (simulation-only) repo, built for a
collaborator who wants to train a **neural-network controller for a grid-forming
converter (GFM)** to improve frequency response.*

---

### What this notebook covers

1. **The repo in 3 lines** — how a system is described and simulated.
2. **A worked example** — a 3-bus network with a synchronous machine, a droop
   **grid-forming converter**, and a small **load step**, run with **dynamic
   line models** and **no state limiters**.
3. **Reading the results** — the converter's *frequency response* to the load step.
4. **Under the hood** — the differential-algebraic-equation (DAE) model and the
   **CasADi symbolic objects** it is built from.
5. **A differentiable plant model** `F(z) → ż` with **exact Jacobians** (CasADi
   autodiff).
6. **Adding the neural control signal** — a residual the network injects into the droop.
7. **CasADi → PyTorch** — wrapping the (stiff-safe, differentiable) plant as a
   `torch.autograd.Function` so you can back-propagate a frequency-deviation loss
   into a neural controller.

### The control we want to learn

A droop grid-forming converter sets its own frequency from an active-power droop:

$$\omega_c \;=\; \omega_c^\star \;+\; R_c^p\,\bigl(p_c^\star - \tilde p_c\bigr)
   \qquad\bigl(\;\omega_c^\star=\omega_{net},\;\; R_c^p=K_p,\;\; p_c^\star=P_{ref}\;\bigr)$$

where $\tilde p_c$ is the (low-pass-filtered) measured active power. Rather than
*replacing* this proven droop, we let a neural network add a corrective
**power-setpoint signal $\Delta p_c^\star$** on top of it:

$$\boxed{\;\omega_c \;=\; \omega_c^\star \;+\; R_c^p\,\bigl(p_c^\star + \Delta p_c^\star - \tilde p_c\bigr),
   \qquad \Delta p_c^\star = \pi_\theta(\text{measurements})\;}$$

The droop keeps charge of the baseline response (and stability); the network only
*augments* it. With $\Delta p_c^\star = 0$ the plant is **exactly** the original
droop converter — so the learned signal is a pure residual, trained by
differentiating through the power-system dynamics.

## 0. Setup

This notebook needs the repo installed (`pip install -e .` from the repo root)
plus `numpy`, `casadi`, and `matplotlib` (all already dependencies). The final
**PyTorch** section additionally needs `pip install torch` — every other cell
runs without it.

In [ ]:
%matplotlib inline
import numpy as np
import casadi as ca
import matplotlib.pyplot as plt
import hermess

print("hermess:", hermess.__file__)
print("casadi:", ca.__version__, "| numpy:", np.__version__)

## 1. The 3-bus test system

```
        bus 1 ───────(line)─────── bus 2 ───────(line)─────── bus 3
          │                          │                          │
        SG1                        load                       GFMI2
   synchronous gen.            ZIP (impedance)         grid-forming converter
        (slack)                                        (sets its own frequency)
          └──────────────────────(line)────────────────────────┘
```

A system is defined by two plain-text tables inside a folder under
`hermess/systems/`:

* **`sim_param.txt`** — devices (machines, converters, loads), network lines, and
  the bus types used for the initial power flow.
* **`sim_dist.txt`** — the disturbances (faults, line switching, **load steps**).

For this tutorial we use `3bus_loadstep`: a synchronous machine at bus 1 (the
slack), a droop grid-forming converter at bus 3, a constant-impedance load at
bus 2, and a small **+10 MW load step at t = 1 s**. Let's look at the two files.

In [ ]:
import pathlib
sysdir = pathlib.Path(hermess.__file__).parent / "systems" / "3bus_loadstep"

print("==================  sim_param.txt  ==================\n")
print(sysdir.joinpath("sim_param.txt").read_text())
print("==================  sim_dist.txt   ==================\n")
print(sysdir.joinpath("sim_dist.txt").read_text())

## 2. Configure and run a simulation

Everything about *how* to simulate (which system, time step, solver, reference
frame, what to model) lives in a `Config` object. Start from the shipped default
and override what you need with `config.updated(**kwargs)`.

The flags that matter for this study:

| flag | value | meaning |
|------|-------|---------|
| `testsystemfile` | `"3bus_loadstep"` | which system folder to load |
| `line_dyn` | **`True`** | **dynamic line models** — transmission lines carry their own differential states (RL+shunt), instead of an algebraic admittance network |
| `incl_lim` | **`False`** | **state limiters off** — no anti-windup / saturation clipping (faster, smooth, fully differentiable) |
| `skip_disturance` | `False` | apply the disturbances in `sim_dist.txt` |
| `omega_mode` | `"nom"` | reference frame rotates at the fixed nominal frequency, so device frequencies move *relative to nominal* — exactly what we want to watch |
| `int_scheme_sim` | `"idas"` | implicit (BDF) DAE solver — needed because the line/filter dynamics are **stiff** |

We set `plot=False` and `small_signal_analysis=False` because we'll do our own
plotting from the returned object (the built-in `plot=True` path pops blocking
Matplotlib windows, which is awkward inside a notebook).

In [ ]:
from hermess.run import run
from hermess.config import config

BASE = dict(
    testsystemfile="3bus_loadstep",
    omega_mode="nom",          # reference frame at nominal frequency
    omega_single_idx=None,
    fn=50,                     # 50 Hz system
    Sb=100,                    # base power [MVA]
    ts=0.001,                  # output time step [s]
    T_start=0.0,
    int_scheme_sim="idas",     # implicit DAE solver (the model is stiff)
    int_scheme_sim_options={
        "reltol": 1e-8, "abstol": 1e-10,
        "max_num_steps": 100000,
        "jit": False,          # set True to JIT-compile the integrator (needs a C compiler)
    },
    line_dyn=True,             # <-- dynamic line models
    incl_lim=False,            # <-- no state limiters
    print_power_flow=False,
    small_signal_analysis=False,
    plot=False, plot_voltage=False, plot_diff=False,
    log_level="WARNING",
)

sim_cfg = config.updated(**BASE, T_end=5.0, skip_disturance=False)
dae = run(sim_cfg)          # returns a DaeSim object holding the model + trajectories
print("\nReturned:", type(dae).__name__)

### What just happened

`run()` parsed the system, built a symbolic DAE, found the steady-state operating
point (initial power flow + device initialization), and time-stepped the implicit
solver through the load step. The returned `DaeSim` object holds **both** the
symbolic model **and** the resulting trajectories.

The state vector splits into three blocks:

* **`x`** — the `nx` device differential states (machine + converter internals),
* **`y`** — the `ny` network-bus voltages (real/imag per bus); differential here
  because `line_dyn=True`,
* **`xl`** — the `nl` line-current states (2 per line).

In [ ]:
print(f"device states   nx = {dae.nx}")
print(f"bus voltages     ny = {dae.ny}   (= 2 x {dae.grid.nn} buses)")
print(f"line currents    nl = {dae.nl}   (= 2 x {dae.grid.nb} lines)")
print(f"private algebraics n_priv = {dae.n_priv}")
print(f"\ndevices: {[d.__class__.__name__ for d in dae.device_list]}")
print(f"\ntrajectory arrays: x_full {dae.x_full.shape}, y_full {dae.y_full.shape}")
print("\nfirst few state names:")
for nm in dae.states[:6]:
    print("   ", nm)

## 3. The frequency response to the load step

When the load steps up by 10 MW at t = 1 s, the grid is briefly short of power.
Both the synchronous machine and the grid-forming converter slow down, and the
converter ramps up its power injection to help — that transient is the *frequency
response* we ultimately want a neural controller to shape.

Each device stores its own trajectories in `device.xf[state_name]`, an array of
shape `(n_units, n_timesteps)`. We pull a couple of helpers out and plot:

* the **synchronous-machine speed** `omega` (an *absolute* p.u. speed, ≈ 1.0),
* the **converter frequency** reconstructed from the droop law
  $\omega_c = \omega_{net} + K_p (P_{ref} - \tilde P_c)$.

In [ ]:
def get_device(dae, class_name):
    return next(d for d in dae.device_list if d.__class__.__name__ == class_name)

sg  = get_device(dae, "SynchronousSubtransientSP")
gfm = get_device(dae, "GridForming")
t   = dae.time_steps

# Synchronous machine speed -> Hz  (omega is absolute p.u. speed, not a deviation)
f_sg = sg.xf["omega"][0] * dae.fn

# Converter frequency from the droop law, then -> Hz
Pc_tilde = gfm.xf["Pc_tilde"][0]
omega_c  = dae.omega_net + float(gfm.Kp[0]) * (float(gfm.Pref[0]) - Pc_tilde)
f_gfm    = omega_c * dae.fn

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(t, f_sg,  label="synchronous machine (bus 1)", lw=2)
ax.plot(t, f_gfm, label="grid-forming converter (bus 3)", lw=2)
ax.axvline(1.0, color="k", ls="--", lw=1, alpha=0.6, label="load step (+10 MW)")
ax.set_xlabel("time [s]"); ax.set_ylabel("frequency [Hz]")
ax.set_title("Frequency response to a small load step")
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout()

In [ ]:
# Converter active-power response and bus voltages
fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4.5))

# Pc_tilde is in p.u. on the converter base (Sn). Convert to MW.
a1.plot(t, Pc_tilde * float(gfm.Sn[0]), color="C2", lw=2)
a1.axvline(1.0, color="k", ls="--", lw=1, alpha=0.6)
a1.set_xlabel("time [s]"); a1.set_ylabel("converter active power [MW]")
a1.set_title("GFM power injection (filtered $\\tilde P_c$)")
a1.grid(alpha=0.3)

for node in dae.grid.buses:
    v_re, v_im = dae.grid.yf[node][0], dae.grid.yf[node][1]
    a2.plot(t, np.hypot(v_re, v_im), label=f"bus {node}", lw=2)
a2.axvline(1.0, color="k", ls="--", lw=1, alpha=0.6)
a2.set_xlabel("time [s]"); a2.set_ylabel("voltage magnitude [p.u.]")
a2.set_title("Bus voltages"); a2.legend(); a2.grid(alpha=0.3)
plt.tight_layout()

The converter frequency dips by a few hundredths of a Hz, then recovers as it
picks up extra load — a textbook droop response. **A learned controller would
reshape this curve**: less dip, faster recovery, better damping, while respecting
the converter's limits. To train one by gradient descent we need a *differentiable*
model of the plant — which is exactly what CasADi gives us.

## 4. Under the hood — the DAE and its CasADi objects

The simulator builds the model symbolically with [CasADi](https://web.casadi.org/).
After `run()`, the `DaeSim` object carries the symbolic ingredients:

| attribute | symbol | what it is |
|-----------|--------|------------|
| `dae.x`   | $x$  | device differential states (`SX`, length `nx`) |
| `dae.y`   | $y$  | bus voltages (`SX`, length `ny`) |
| `dae.xl`  | $i_\ell$ | line-current states (`SX`, length `nl`) |
| `dae.f`   | $\dot x$ | RHS of the device ODEs |
| `dae.fnode` | $\dot y$ | RHS of the bus-voltage ODEs (capacitor balance) |
| `dae.fl`  | $\dot i_\ell$ | RHS of the line-current ODEs |
| `dae.g`   | $g$  | algebraic equations (empty here — the whole network is differential under `line_dyn=True`) |

Two substitutions turn these into a closed-form vector field:

1. **Reference frequency.** The equations are written against symbolic placeholders
   `omega_ref*`; the actual reference-frame expressions live in `omega_ref*_expr`.
2. **Switches.** `dae.s` / `dae.sl` are limiter/line on-off parameters; with
   `incl_lim=False` they're all ones.

Each device also exposes the *global indices* of its states as attributes, so you
can find any state in the stacked vector. For the converter, the droop expression
itself is stored symbolically on `gfm.omega_c`.

In [ ]:
print("dae.x  :", type(dae.x).__name__,  dae.x.shape)
print("dae.f  :", type(dae.f).__name__,  dae.f.shape)
print("dae.g  :", type(dae.g).__name__,  dae.g.shape, " (empty -> fully differential network)")

print("\nGrid-forming converter states (global indices into x):")
for s in gfm.states:
    print(f"   {s:10s} -> x[{int(getattr(gfm, s)[0]):2d}]")

print("\nThe droop law, symbolically  (gfm.omega_c):")
print("   omega_c =", gfm.omega_c)
print("\nThe angle-state ODE that consumes it  (dae.f[delta_c]):")
print("   d(delta_c)/dt =", dae.f[int(gfm.delta_c[0])])

## 5. A differentiable plant model $F(z)\to\dot z$

We stack the full state $z = [\,x;\;y;\;i_\ell\,]$ and assemble the continuous-time
vector field $\dot z = F(z)$ as a CasADi `Function`. CasADi then gives us the exact
Jacobian $\partial F/\partial z$ for free by automatic differentiation — the
backbone of any gradient-based control design.

**One important subtlety.** After a run *with* a disturbance, the symbolic
equations reflect the **post-disturbance** system (the load step is baked in), so
`F` evaluated at the pre-step start point is *not* at equilibrium. To get a clean
nominal plant we rebuild the model with `skip_disturance=True`. (You then apply
disturbances yourself — e.g. by stepping a load parameter — which is exactly what
you want for controller training anyway.)

In [ ]:
def build_rhs(dae):
    '''Assemble the continuous-time RHS  z -> zdot  for a line_dyn=True model.

    Returns (z, rhs) as CasADi SX, with z = [x; y; xl]. Performs the same
    reference-frequency and switch substitutions that the integrator does
    internally in DaeSim.fgcall().
    '''
    W_sym  = ca.vertcat(dae.omega_ref,       dae.omega_ref_buses,       dae.omega_ref_lines)
    W_expr = ca.vertcat(dae.omega_ref_expr,  dae.omega_ref_buses_expr,  dae.omega_ref_lines_expr)
    s_sym  = ca.vertcat(dae.s, dae.sl)
    s_val  = ca.vertcat(ca.DM(dae.sinit), ca.DM(dae.slinit))   # incl_lim=False -> all ones

    def sub(e):
        e = ca.substitute(e, W_sym, W_expr)
        return ca.substitute(e, s_sym, s_val)

    f, fnode, fl = sub(dae.f), sub(dae.fnode), sub(dae.fl)
    z   = ca.vertcat(dae.x, dae.y, dae.xl)
    rhs = ca.vertcat(f, fnode, fl)
    return z, rhs

# Clean nominal plant (no disturbance baked in). A short horizon is enough; we
# only need the symbolic model + the equilibrium initial point, not a trajectory.
model = run(config.updated(**BASE, T_end=1.0, skip_disturance=True))
gfm_m = get_device(model, "GridForming")

z, rhs = build_rhs(model)
F = ca.Function("F", [z], [rhs], ["z"], ["zdot"])
J = ca.Function("J", [z], [ca.jacobian(rhs, z)], ["z"], ["dF_dz"])

z0 = np.concatenate([model.xinit, model.yinit, model.xlinit])
print("state dimension:", z.size1(), " (= nx+ny+nl =", model.nx, "+", model.ny, "+", model.nl, ")")
print("||F(z0)||_inf =", float(ca.norm_inf(F(z0))), "  -> ~0, confirms z0 is an equilibrium")
print("Jacobian shape:", J(z0).shape)

In [ ]:
# Sanity-check the exact (autodiff) Jacobian against finite differences.
zt = z0 + 0.01                       # perturb off equilibrium so the test is non-trivial
eps = 1e-6
f0  = np.asarray(F(zt)).ravel()
Jfd = np.zeros((z.size1(), z.size1()))
for j in range(z.size1()):
    dz = np.zeros_like(zt); dz[j] = eps
    Jfd[:, j] = (np.asarray(F(zt + dz)).ravel() - f0) / eps
Jad = np.asarray(J(zt))
print("max relative error (autodiff vs finite-diff Jacobian):",
      np.max(np.abs(Jad - Jfd)) / (np.max(np.abs(Jad)) + 1e-12))

# Stiffness: the line/filter modes are fast -> use an implicit integrator.
ev = np.linalg.eigvals(np.asarray(J(z0)))
print(f"|eig(J)| up to {np.max(np.abs(ev)):.0f} rad/s  "
      f"->  explicit Euler is stable only for dt < {2/np.max(np.abs(ev)):.1e} s")

## 6. Adding the neural control signal $\Delta p_c^\star$

We keep the droop and inject the network's residual **power-setpoint signal**
$\Delta p_c^\star$ inside it:

$$\omega_c \;=\; \omega_c^\star + R_c^p\,\bigl(p_c^\star + \Delta p_c^\star - \tilde p_c\bigr).$$

The converter frequency enters the dynamics only through the angle ODE
$\dot{\delta_c} = \omega_b\,(\omega_c - \omega_{ref})$, so we rebuild that single
row with the augmented $\omega_c$ and expose $\Delta p_c^\star$ as the control
input. Everything else — including the power-measurement state $\tilde p_c$ the
controller reads — is untouched, and $\Delta p_c^\star = 0$ reproduces the original
droop plant exactly.

In [ ]:
omega_b   = 2 * np.pi * model.fn          # base angular frequency [rad/s]
omega_ref = model.omega_net                # = 1.0 p.u. in 'nom' reference frame
n_u       = gfm_m.n                         # one control signal per converter

dPc = ca.SX.sym("dPc", n_u)                 # NN power-setpoint augmentation  Δp_c*  [p.u.]
rhs_ctrl = ca.SX(rhs)                        # copy of the autonomous (droop) RHS
for k in range(n_u):
    Pc_tilde_k = model.x[int(gfm_m.Pc_tilde[k])]            # filtered measured power  p̃_c
    omega_c = model.omega_net + float(gfm_m.Kp[k]) * (
        float(gfm_m.Pref[k]) + dPc[k] - Pc_tilde_k)          # augmented droop law
    rhs_ctrl[int(gfm_m.delta_c[k])] = omega_b * (omega_c - omega_ref)

F_ctrl = ca.Function("F_ctrl", [z, dPc], [rhs_ctrl], ["z", "dPc"], ["zdot"])

# Δp_c* = 0 must reproduce the original droop plant exactly (pure residual).
print("F_ctrl(z0, 0) vs F(z0):", float(ca.norm_inf(F_ctrl(z0, np.zeros(n_u)) - F(z0))),
      " -> 0, good (Δp_c* is a residual on top of the droop)")
print("\naugmented angle ODE  d(delta_c)/dt =", rhs_ctrl[int(gfm_m.delta_c[0])])

### The production-grade route: a pluggable control strategy

The symbolic surgery above is perfect for experiments. For a *permanent* learned
controller, the cleaner path is to use the codebase's extension point: the
converter's angle behaviour is a swappable **`AngleSource`** strategy in
`hermess/devices/inverter_angle.py`. The shipped droop is just:

```python
class DroopAngle(_PowerDroopAngle):
    def fgcall(self, host, dae, omega_ref_vec, omega_b):
        host.omega_c = dae.omega_net + host.Kp * (host.Pref - dae.x[host.Pc_tilde])
        dae.f[host.delta_c] = omega_b * (host.omega_c - omega_ref_vec)
        return host.omega_c
```

A `NeuralAngle(AngleSource)` would add the network's $\Delta p_c^\star$ inside the
**same** droop — `host.Kp * (host.Pref + dPc_nn - dae.x[host.Pc_tilde])` — and
nothing else in the simulator would change.

## 7. From CasADi to PyTorch

Three ways to get the model into a PyTorch training loop:

1. **Wrap the CasADi function as a `torch.autograd.Function`** — call CasADi for the
   value (forward) and for the exact Jacobian (backward). Simplest, exact gradients,
   works today. *(shown below)*
2. **Re-implement the symbolic graph in pure PyTorch** — fully on-device/GPU and
   fusable, at the cost of porting the math. Tools like
   [`l4casadi`](https://github.com/Tim-Salzmann/l4casadi) automate this.
3. **Export C code** via `Function.generate(...)` and bind it.

We use option 1, and we wrap a **single implicit integration step**
$z_{k+1} = \Phi(z_k, \Delta p^\star_{c,k})$ rather than the bare RHS. Why: the plant
is stiff (see §5), so an explicit `z + dt·F` step would need `dt < 1.6e-4 s`. A
CasADi `integrator` takes one stable implicit step over `dt` **and** exposes exact
sensitivities $\partial z_{k+1}/\partial z_k$ and
$\partial z_{k+1}/\partial \Delta p^\star_{c,k}$ — precisely the Jacobians the
backward pass needs.

In [ ]:
# One stable, differentiable implicit step  (z_k, Δp_c*) -> z_{k+1}, built with CasADi.
dt = 0.005
step = ca.integrator("step", "idas", {"x": z, "p": dPc, "ode": rhs_ctrl}, 0.0, dt,
                     {"reltol": 1e-8, "abstol": 1e-10})
z_next = step(x0=z, p=dPc)["xf"]

StepF  = ca.Function("StepF",  [z, dPc], [z_next])                  # value
StepJz = ca.Function("StepJz", [z, dPc], [ca.jacobian(z_next, z)])    # dz_{k+1}/dz_k
StepJu = ca.Function("StepJu", [z, dPc], [ca.jacobian(z_next, dPc)])  # dz_{k+1}/d(Δp_c*)_k

no_ctrl = np.zeros(n_u)                       # Δp_c* = 0  ->  the plain droop converter
print("one implicit step ok;  StepJz:", StepJz(z0, no_ctrl).shape,
      " StepJu:", StepJu(z0, no_ctrl).shape)
print("Δp_c*=0 step is a no-op (||z_next - z0||):",
      float(ca.norm_inf(StepF(z0, no_ctrl) - z0)))

In [ ]:
# Verify the bridge math in numpy (this is exactly what torch's backward computes):
# the vector-Jacobian product  g -> J^T g.  Always runs, no torch needed.
g = np.random.default_rng(0).standard_normal(z.size1())
vjp_z   = np.asarray(StepJz(z0, no_ctrl)).T @ g
vjp_dPc = np.asarray(StepJu(z0, no_ctrl)).T @ g
print("VJP shapes:", vjp_z.shape, vjp_dPc.shape, " (= grads wrt z_k and Δp_c*_k)")
print("These J^T g products are what GfmPlantStep.backward returns below.")

In [ ]:
# ---- PyTorch bridge ----  (needs `pip install torch`; otherwise this cell no-ops)
try:
    import torch

    class GfmPlantStep(torch.autograd.Function):
        '''Differentiable one-step rollout of the GFM power-system plant.

        forward : z_{k+1} = StepF(z_k, dPc_k)            (CasADi implicit integrator)
        backward: grad_z   = StepJz^T grad_out
                  grad_dPc = StepJu^T grad_out
        dPc_k is the NN residual power-setpoint signal Δp_c*. Single-sample (no
        batch dim); loop or vmap over a batch.
        '''
        @staticmethod
        def forward(ctx, z_k, dPc_k):
            z_np = z_k.detach().cpu().numpy().reshape(-1, 1)
            u_np = dPc_k.detach().cpu().numpy().reshape(-1, 1)
            z1 = np.asarray(StepF(z_np, u_np)).reshape(-1)
            ctx.save_for_backward(z_k, dPc_k)
            return torch.as_tensor(z1, dtype=z_k.dtype, device=z_k.device)

        @staticmethod
        def backward(ctx, grad_out):
            z_k, dPc_k = ctx.saved_tensors
            z_np = z_k.detach().cpu().numpy().reshape(-1, 1)
            u_np = dPc_k.detach().cpu().numpy().reshape(-1, 1)
            g = grad_out.detach().cpu().numpy().reshape(-1)
            gz   = np.asarray(StepJz(z_np, u_np)).T @ g
            gdPc = np.asarray(StepJu(z_np, u_np)).T @ g
            to = lambda a: torch.as_tensor(a, dtype=grad_out.dtype, device=grad_out.device)
            return to(gz), to(gdPc)

    # --- tiny differentiable-simulation training skeleton ---
    # Task: after a small disturbance, damp the excursion by learning the residual
    # signal Δp_c* on top of the droop. (Δp_c* = 0 leaves the plain droop converter,
    # which already holds z0 at rest, so penalizing the excursion from z0 is a
    # clean, well-posed objective.)
    nz = z.size1()
    torch.manual_seed(0)
    controller = torch.nn.Sequential(
        torch.nn.Linear(nz, 32), torch.nn.Tanh(), torch.nn.Linear(32, n_u)
    ).double()                                   # run the whole plant in float64
    # (zero-init the last layer to start from exactly pure droop, Δp_c* ≈ 0)
    opt = torch.optim.Adam(controller.parameters(), lr=1e-3)

    z_eq   = torch.tensor(z0)                     # nominal equilibrium
    pc_idx = [int(i) for i in gfm_m.Pc_tilde]

    z_pert = z_eq.clone()                         # perturb the measured-power state
    for i in pc_idx:
        z_pert[i] = z_pert[i] + 0.1

    def rollout_loss(horizon=40):
        z_t  = z_pert.clone()
        loss = z_t.new_zeros(())
        for _ in range(horizon):
            dPc_t = controller(z_t)               # NN residual power-setpoint signal Δp_c*
            z_t   = GfmPlantStep.apply(z_t, dPc_t)   # one stable, differentiable plant step
            loss  = loss + ((z_t - z_eq) ** 2).sum()   # penalize excursion from equilibrium
        return loss

    print("initial rollout loss (pure droop + untrained residual):", float(rollout_loss().detach()))
    for it in range(5):
        opt.zero_grad()
        L = rollout_loss()
        L.backward()                      # gradients flow back through the power-system plant
        opt.step()
        print(f"  step {it}: loss = {float(L):.6f}")
    print("Loss is differentiable wrt the controller weights -> ready to train a real policy.")

except ImportError:
    print("PyTorch not installed - skipping the live demo.")
    print("Install it (`pip install torch`) and re-run this cell to train the controller.")
    print("The GfmPlantStep autograd.Function above is the complete CasADi<->PyTorch bridge.")

## 8. Where to go next

* **Tune the experiment.** Edit `systems/3bus_loadstep/sim_param.txt` (e.g. the
  droop gain `Kp`, converter rating `Sn`, filter/line parameters) and
  `sim_dist.txt` (size/timing of the load step, or add faults / line switching).
* **The control signal.** §6 injects the residual $\Delta p_c^\star$ into the droop
  by symbolic surgery; for a permanent learned controller, add a
  `NeuralAngle(AngleSource)` strategy in `devices/inverter_angle.py` (see §6).
* **Stiffness vs. speed.** `line_dyn=True` keeps the fast network/filter modes
  (realistic, but stiff). For faster controller training you can set
  `line_dyn=False` for a quasi-static (algebraic) network — a smaller, less stiff
  model — at the cost of dropping the fast electromagnetic transients. (Note: the
  `build_rhs` helper here assumes `line_dyn=True`; the algebraic case puts the
  network in `dae.g` instead of `dae.fnode`/`dae.fl`.)
* **Batching / GPU.** `GfmPlantStep` is single-sample; loop or `torch.vmap` it for
  a batch, or move to a pure-PyTorch / `l4casadi` reimplementation when CasADi calls
  become the bottleneck.
* **Reference frames & analysis.** Try `omega_mode="coi"` (centre of inertia) and
  flip `small_signal_analysis=True` for eigenvalue / participation-factor reports at
  the operating point.

**Recap of the bridge:** `run()` → symbolic `DaeSim` → `build_rhs` → CasADi
`Function` `F(z)` (+ exact Jacobian) → inject the residual $\Delta p_c^\star$ into
the droop → wrap a stable implicit step as a `torch.autograd.Function`. From there
it's a standard PyTorch training loop.